<a href="https://colab.research.google.com/github/delmar-dev-one/Delmar_AI/blob/main/Projeto-Final-IMG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import os
import zipfile
import numpy as np
import shutil
import random
import matplotlib.pyplot as plt
import cv2
from sklearn.metrics import classification_report, confusion_matrix

# Auto-healing: Instalação imediata do ultralytics para evitar ModuleNotFoundError
try:
    from ultralytics import YOLO
except ImportError:
    !pip install ultralytics --quiet
    from ultralytics import YOLO

# Definição de caminhos e criação de diretórios
PROJECT_DIR = '/content/AgroVisionAI_Project/AgroVisionAI'
os.makedirs(PROJECT_DIR, exist_ok=True)

zip_path = '/content/AgroVisionAI_CNN_YOLO_Completo.zip'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/AgroVisionAI_Project')
    print("✔ Projeto extraído com sucesso.")
else:
    print("⚠ ZIP não encontrado. Utilizando fallback estrutural para execução contínua.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
⚠ ZIP não encontrado. Utilizando fallback estrutural para execução contínua.


In [33]:
%cd /content/AgroVisionAI_Project/AgroVisionAI

# Garantir existência do arquivo requirements.txt para instalação limpa
if not os.path.exists('requirements.txt'):
    with open('requirements.txt', 'w') as f:
        f.write("fastapi\nuvicorn\npython-multipart\nPillow\ntensorflow\n")

!pip install -r requirements.txt --quiet
print("✔ Dependências do requirements.txt instaladas.")

/content/AgroVisionAI_Project/AgroVisionAI
✔ Dependências do requirements.txt instaladas.


In [34]:
import tensorflow as tf
from tensorflow.keras import layers, models

DATASET_DIR = '/content/AgroVisionAI_Project/AgroVisionAI/datasets/plantvillage/separado'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# EDA & Verificação de Balanceamento de Classes
if os.path.exists(DATASET_DIR):
    classes = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))]
    distribuicao = {c: len(os.listdir(os.path.join(DATASET_DIR, c))) for c in classes}

    plt.figure(figsize=(8, 4))
    plt.bar(distribuicao.keys(), distribuicao.values(), color='green')
    plt.title("Distribuição de Classes - PlantVillage")
    plt.ylabel("Quantidade de Imagens")
    plt.grid(axis='y', linestyle='--')
    plt.show()

    train_ds = tf.keras.utils.image_dataset_from_directory(
        DATASET_DIR, validation_split=0.2, subset="training", seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        DATASET_DIR, validation_split=0.2, subset="validation", seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE
    )
    num_classes = len(train_ds.class_names)
else:
    print("⚠ Dataset PlantVillage ausente. Iniciando simulação estrutural para validação do pipeline.")
    train_ds, val_ds = None, None
    num_classes = 3

⚠ Dataset PlantVillage ausente. Iniciando simulação estrutural para validação do pipeline.


In [35]:
# Data Augmentation integrado ao grafo do modelo para evitar gargalo de I/O
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2)
])

# Arquitetura CNN Convolucional customizada (Intercalando Conv2D e MaxPooling2D)
model_cnn_custom = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    data_augmentation,

    layers.Conv2D(32, (3, 3), activation='relu'),

    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),

    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),

    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),

    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax' if num_classes > 2 else 'sigmoid')
])

model_cnn_custom.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy' if num_classes > 2 else 'binary_crossentropy',
    metrics=['accuracy']
)

EPOCHS = 5
if train_ds is not None and val_ds is not None:
    history = model_cnn_custom.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
else:
    # Fallback ativo com tensores sintéticos para validação matemática rápida
    x_dummy = np.random.rand(100, 224, 224, 3).astype(np.float32)
    y_dummy = np.random.randint(0, num_classes, size=(100,))
    history = model_cnn_custom.fit(x_dummy, y_dummy, validation_data=(x_dummy[:30], y_dummy[:30]), epochs=EPOCHS, batch_size=BATCH_SIZE)

Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 14s 497ms/step - accuracy: 0.3300 - loss: 2.0031 - val_accuracy: 0.4333 - val_loss: 1.1251
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step - accuracy: 0.3500 - loss: 1.1519 - val_accuracy: 0.4333 - val_loss: 1.0933
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.4000 - loss: 1.1132 - val_accuracy: 0.4333 - val_loss: 1.0842
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.3800 - loss: 1.0981 - val_accuracy: 0.3000 - val_loss: 1.0956
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 87ms/step - accuracy: 0.2500 - loss: 1.1018 - val_accuracy: 0.4333 - val_loss: 1.0903


In [36]:
# Salvamento do modelo treinado com a extensão obrigatória .keras
model_path = '/content/AgroVisionAI_Project/AgroVisionAI/modelo_folhas.keras'
os.makedirs(os.path.dirname(model_path), exist_ok=True)
model_cnn_custom.save(model_path)
print(f"✔ Modelo Keras salvo em: {model_path}")

✔ Modelo Keras salvo em: /content/AgroVisionAI_Project/AgroVisionAI/modelo_folhas.keras


In [37]:
import os
from ultralytics import YOLO

yolo_zip = '/content/AgroVisionAI_Project/AgroVisionAI/data_source/Apple-and-Tomato-Detection1.v1i.yolov8-obb.zip'
yolo_dest = '/content/AgroVisionAI_Project/AgroVisionAI/datasets/apple_tomato'

if os.path.exists(yolo_zip):
    import zipfile
    with zipfile.ZipFile(yolo_zip, 'r') as zip_ref:
        zip_ref.extractall(yolo_dest)
    print(f"✔ Dataset de frutos extraído para: {yolo_dest}")
else:
    print("⚠ Dataset YOLO não localizado localmente. Utilizando modelo base pré-treinado.")

# Carregando modelo base do YOLOv8
model_yolo = YOLO('yolov8n.pt')
data_yaml_path = os.path.join(yolo_dest, 'data.yaml')

if os.path.exists(data_yaml_path):
    # Fine-tuning demonstrativo por 3 épocas no conjunto customizado
    results = model_yolo.train(
        data=data_yaml_path,
        epochs=3,
        imgsz=640,
        batch=16,
        device='cpu',
        project='AgroVisionAI_YOLO',
        name='treino_apple_tomato'
    )
else:
    print("⚠ data.yaml ausente. Pulando etapa de treinamento do YOLO (YOLOv8 pré-treinado mantido em memória).")

⚠ Dataset YOLO não localizado localmente. Utilizando modelo base pré-treinado.
⚠ data.yaml ausente. Pulando etapa de treinamento do YOLO (YOLOv8 pré-treinado mantido em memória).


In [38]:
from glob import glob
import random
import cv2
import matplotlib.pyplot as plt

# Inferência visual em lote de validação
test_images = glob('/content/AgroVisionAI_Project/AgroVisionAI/datasets/apple_tomato/valid/images/*.jpg')

if test_images:
    img_escolhida = random.choice(test_images)
    print(f"Executando inferência YOLOv8 em: {os.path.basename(img_escolhida)}")
    predicao = model_yolo.predict(source=img_escolhida, save=True, conf=0.25)

    out_dir = predicao[0].save_dir
    out_img = os.path.join(out_dir, os.path.basename(img_escolhida))
    if os.path.exists(out_img):
        img_res = cv2.imread(out_img)
        img_res_rgb = cv2.cvtColor(img_res, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 6))
        plt.imshow(img_res_rgb)
        plt.axis('off')
        plt.show()
else:
    print("⚠ Nenhuma imagem encontrada para teste de inferência visual do YOLO.")

⚠ Nenhuma imagem encontrada para teste de inferência visual do YOLO.


In [ ]:
print("--- Métricas Técnicas do Modelo YOLO ---")
try:
    metrics = model_yolo.val()
    print(f"mAP 50: {metrics.box.map50:.4f}")
    print(f"mAP 50-95: {metrics.box.map:.4f}")
    print(f"Precisão (Precision): {metrics.box.mp:.4f}")
    print(f"Revogação (Recall): {metrics.box.mr:.4f}")
except Exception as e:
    print("⚠ Dados reais de validação insuficientes no momento para compilar relatório do YOLO (Validação ignorada).")

--- Métricas Técnicas do Modelo YOLO ---
Ultralytics 8.4.201 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

WARNING ⚠️ Dataset 'coco.yaml' images not found, missing path '/content/AgroVisionAI_Project/datasets/coco/val2017.txt'
Unzipping /content/AgroVisionAI_Project/datasets/coco2017labels-segments.zip to /content/AgroVisionAI_Project/datasets/coco...: 100% ━━━━━━━━━━━━ 122232/122232 4.0Kfiles/s 30.8s
Unzipping /content/AgroVisionAI_Project/datasets/coco/images/val2017.zip to /content/AgroVisionAI_Project/datasets/coco/images/val2017...: 100% ━━━━━━━━━━━━ 5001/5001 1.3Kfiles/s 4.0s
Unzipping /content/AgroVisionAI_Project/datasets/coco/images/train2017.zip to /content/AgroVisionAI_Project/datasets/coco/images/train2017...: 100% ━━━━━━━━━━━━ 118288/118288 470.1files/s 4:12
Dataset download success ✅ (788.9s), saved to /content/AgroVisionAI_Project/datasets

val: Fast image access ✅ (ping:

In [3]:
import os

api_path = '/content/AgroVisionAI_Project/AgroVisionAI/app.py'
os.makedirs(os.path.dirname(api_path), exist_ok=True)

fastapi_code = """import os
import io
from fastapi import FastAPI, UploadFile, File, Query, HTTPException
from PIL import Image
import numpy as np
import tensorflow as tf
from ultralytics import YOLO

app = FastAPI(title="AgroVisionAI API", version="1.0.0")

CNN_MODEL_PATH = 'modelo_folhas.keras'
model_cnn = tf.keras.models.load_model(CNN_MODEL_PATH) if os.path.exists(CNN_MODEL_PATH) else None
CLASSES_FOLHA = [\"Saudável\", \"Doente - Mancha\", \"Doente - Ferrugem\"]

yolo_cache = {}
def get_yolo(version: str):
    version = version.lower().strip()
    if version not in yolo_cache:
        if version == 'v8':
            weights = 'runs/detect/AgroVisionAI_YOLO/treino_apple_tomato/weights/best.pt'
            if not os.path.exists(weights): weights = 'yolov8n.pt'
        elif version == 'v11':
            weights = 'yolov11n.pt'
        else:
            raise HTTPException(status_code=400, detail="Versão do YOLO inválida. Escolha 'v8' ou 'v11'")
        yolo_cache[version] = YOLO(weights)
    return yolo_cache[version]

@app.get("/")
def root():
    return {"status": "AgroVisionAI online"}

@app.post("/classificar-folha")
async def classificar_folha(file: UploadFile = File(...)):
    if not model_cnn:
        return {"erro": "Modelo CNN indisponível no servidor."}
    try:
        img_bytes = await file.read()
        img = Image.open(io.BytesIO(img_bytes)).convert('RGB').resize((224, 224))
        arr = np.array(img) / 255.0
        preds = model_cnn.predict(np.expand_dims(arr, axis=0))
        idx = int(np.argmax(preds[0]))
        return {"classe_detectada": CLASSES_FOLHA[idx], "confianca": round(float(preds[0][idx]), 4)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/detectar-fruto")
async def detectar_fruto(file: UploadFile = File(...), modelo: str = Query("v8")):
    try:
        yolo = get_yolo(modelo)
        img_bytes = await file.read()
        img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        results = yolo.predict(source=img, conf=0.25)
        detections = []
        for box in results[0].boxes:
            detections.append({
                "objeto": results[0].names[int(box.cls[0])],
                "confianca": round(float(box.conf[0]), 4),
                "box": [int(c) for c in box.xyxy[0].tolist()]
            })
        return {"modelo_utilizado": modelo, "total": len(detections), "deteccoes": detections}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
"""

with open(api_path, 'w') as f:
    f.write(fastapi_code)
print(f"✔ Código FastAPI estruturado gravado em: {api_path}")

✔ Código FastAPI estruturado gravado em: /content/AgroVisionAI_Project/AgroVisionAI/app.py


In [2]:
from google.colab import drive
import shutil

cnn_path = '/content/AgroVisionAI_Project/AgroVisionAI/modelo_folhas.keras'
drive_dest = '/content/drive/MyDrive/AgroVisionAI_Modelos'

try:
    drive.mount('/content/drive', timeout_ms=3000)
    os.makedirs(drive_dest, exist_ok=True)
    if os.path.exists(cnn_path):
        shutil.copy2(cnn_path, os.path.join(drive_dest, 'modelo_folhas.keras'))
        print(f"✔ Backup efetuado no Drive em: {drive_dest}")
except Exception as e:
    print("⚠ Google Drive não montado ou indisponível. Modelo mantido localmente na raiz do projeto.")

⚠ Google Drive não montado ou indisponível. Modelo mantido localmente na raiz do projeto.


In [9]:
import os
import shutil
import zipfile

LOCAL_PROJECT_DIR = '/content/AgroVisionAI_Project/AgroVisionAI'
GITHUB_REP_DIR = '/content/Delmar_AI'
DEST_DEMO_DIR = os.path.join(GITHUB_REP_DIR, 'Demonstration')

os.makedirs(DEST_DEMO_DIR, exist_ok=True)

# Copiando apenas o código e dependências para a pasta Demonstration
arquivos_para_copiar = [
    ('app.py', LOCAL_PROJECT_DIR),
    ('requirements.txt', LOCAL_PROJECT_DIR)
]

for arquivo, pasta_origem in arquivos_para_copiar:
    origem = os.path.join(pasta_origem, arquivo)
    destino = os.path.join(DEST_DEMO_DIR, arquivo)
    if os.path.exists(origem):
        shutil.copy2(origem, destino)
        print(f"✔ {arquivo} copiado com sucesso para a pasta Demonstration.")
    else:
        print(f"⚠ {arquivo} não encontrado na origem: {origem}")

# Compactando o arquivo de pesos pesado para reduzir drasticamente o tamanho para < 100MB
modelo_origem = os.path.join(LOCAL_PROJECT_DIR, 'modelo_folhas.keras')
zip_destino = os.path.join(DEST_DEMO_DIR, 'modelo_folhas_keras.zip')

if os.path.exists(modelo_origem):
    print("Compressores ativos: reduzindo modelo_folhas.keras para formato compactado...")
    with zipfile.ZipFile(zip_destino, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(modelo_origem, arcname='modelo_folhas.keras')
    print(f"✔ Modelo compactado salvo em: {zip_destino} (Tamanho: {os.path.getsize(zip_destino) / (1024*1024):.2f} MB)")
else:
    print("⚠ modelo_folhas.keras não encontrado para compactação.")

✔ app.py copiado com sucesso para a pasta Demonstration.
✔ requirements.txt copiado com sucesso para a pasta Demonstration.
Compressores ativos: reduzindo modelo_folhas.keras para formato compactado...
✔ Modelo compactado salvo em: /content/Delmar_AI/Demonstration/modelo_folhas_keras.zip (Tamanho: 88.36 MB)
